<a href="https://colab.research.google.com/github/parthaborgohain566/Prodigy-Infotech-Internship-Generative-AI-GA/blob/main/Task%2001%3A%20Text%20Generation%20with%20GPT-2/%20Fine_Tune_GPT2_Model_using_tinystories_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Primary Modules

* **`torch`**: Sets up tensor operations and GPU/CPU resource management.
* **`load_dataset`**: Downloads and manages dataset pipelines.
* **`AutoTokenizer`**: Maps text inputs into model-compatible token IDs.
* **`AutoModelForCausalLM`**: Loads autoregressive model architectures (e.g., LLaMA, GPT, Mistral).
* **`Trainer`**: Automates training, evaluation, and logging loops.
* **`TrainingArguments`**: Configures hyperparameters (batch size, learning rate, epochs).
# * **`DataCollatorForLanguageModeling`**: Manages dynamic sequence padding and batching.

In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)


### 2 Pre-trained Artifacts

* **`model_name = "gpt2"`**: Specifies the target model checkpoint to download from Hugging Face.
* **`AutoTokenizer.from_pretrained(...)`**: Instantiates the GPT-2 tokenizer, loading the vocabulary, BPE merge rules, and formatting rules.
* **`AutoModelForCausalLM.from_pretrained(...)`**: Loads the pre-trained weights and standard Causal Language Modeling architecture for text generation.

In [2]:
# 1. Load GPT-2 Model & Tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
# GPT-2 does not have a default pad token; configure pad token to eos_token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

###3 Key Operations

* **`import os`**: Handles file system and directory path operations.
* **`snapshot_download(...)`**: Downloads all repository files (data splits, metadata, configs) in a single step.
* **`repo_id="roneneldan/TinyStories"`**: Targets the TinyStories dataset on Hugging Face.
* **`repo_type="dataset"`**: Specifies that the source target is a dataset repository rather than a model checkpoint.
* **`local_dir=target_dir`**: Forces the files to be saved directly to `/content/tinystories_data`.

In [2]:
import os
from huggingface_hub import snapshot_download

# Define the target folder inside /content/
target_dir = "/content/tinystories_data"

# Download the dataset repository directly to /content/
snapshot_download(
    repo_id="roneneldan/TinyStories",
    repo_type="dataset",
    local_dir=target_dir
)

print(f"Dataset downloaded to: {target_dir}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Dataset downloaded to: /content/tinystories_data


In [7]:
from datasets import load_dataset

# 1. Load the TinyStories dataset
dataset = load_dataset("/content/tinystories_data")

# 2. Define output paths in your /content/ folder
train_file_path = "/content/train100.txt"
test_file_path = "/content/test102.txt"

# 3. Write training data (taking a subset, e.g., 10,000 stories)
print("Writing train.txt...")
with open(train_file_path, "w", encoding="utf-8") as f:
    for item in dataset["train"].select(range(100)):
        # Write story followed by double newline as a story delimiter
        f.write(item["text"].strip() + "\n\n")

# 4. Write testing/validation data (taking a subset, e.g., 1,000 stories)
print("Writing test.txt...")
with open(test_file_path, "w", encoding="utf-8") as f:
    for item in dataset["validation"].select(range(100)):
        f.write(item["text"].strip() + "\n\n")

print(f"Done! Files saved at:\n - {train_file_path}\n - {test_file_path}")

Writing train.txt...
Writing test.txt...
Done! Files saved at:
 - /content/train100.txt
 - /content/test100.txt


### 4 Key Operations

* **`load_clean_stories(...)`**: Reads raw text files, splits them by double line breaks (`\n\n`), filters out empty or ultra-short entries ($\le 5$ characters), and converts them into Hugging Face `Dataset` objects.
* **`train_dataset` & `eval_dataset`**: Instantiates datasets using local training (`train100.txt`) and evaluation (`test102.txt`) text files.
* **`tokenize_function(...)`**: Converts raw text into token IDs with sequence truncation set to a maximum length of 256.
* **`map(...)`**: Applies the tokenizer across both train and evaluation datasets in batches, dropping original raw text columns to save memory.
* **`filter(...)`**: Removes empty token sequences by retaining only samples that contain more than 1 token ID.
* **`DataCollatorForLanguageModeling(..., mlm=False)`**: Prepares dynamic batching and padding for standard autoregressive (causal) language modeling (disabling masked language modeling).

In [4]:
# 3. Tokenize Dataset and Filter Empty Tensors
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

def load_clean_stories(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    # Split by double newline and filter out strings that are empty or just whitespace
    raw_stories = [s.strip() for s in content.split("\n\n") if len(s.strip()) > 5]
    return Dataset.from_dict({"text": raw_stories})

train_dataset = load_clean_stories("/content/train100.txt")
eval_dataset = load_clean_stories("/content/test102.txt")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding=False
    )

# Map tokenization
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Strict filter: keep only examples where input_ids actually contains tokens
tokenized_train = tokenized_train.filter(lambda x: len(x['input_ids']) > 1)
tokenized_eval = tokenized_eval.filter(lambda x: len(x['input_ids']) > 1)

# Data collator for causal language modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Map:   0%|          | 0/429 [00:00<?, ? examples/s]

Map:   0%|          | 0/189 [00:00<?, ? examples/s]

Filter:   0%|          | 0/429 [00:00<?, ? examples/s]

Filter:   0%|          | 0/189 [00:00<?, ? examples/s]

###5 Key Arguments & Hyperparameters

* **`output_dir="./gpt2-tinystories112"`**: Directory path where training checkpoints, logs, and model artifacts are stored.
* **`eval_strategy="epoch"`**: Evaluates the model loss against the validation dataset at the end of every epoch.
* **`learning_rate=5e-5`**: Base learning rate used by the optimizer.
* **`weight_decay=0.01`**: Applies $L_2$ regularization to prevent overfitting.
* **`per_device_train_batch_size=8` & `per_device_eval_batch_size=8`**: Sets micro-batch size per GPU/CPU device for both training and evaluation.
* **`num_train_epochs=3`**: Defines the total number of complete training passes over the dataset.
* **`logging_steps=5`**: Triggers metric and loss output every 5 training steps.
* **`save_strategy="epoch"` & `save_total_limit=2`**: Saves a checkpoint at the end of each epoch, retaining only the 2 most recent checkpoints to conserve storage.
* **`fp16=torch.cuda.is_available()`**: Enables 16-bit floating-point mixed-precision training if a CUDA-supported GPU is available to reduce memory consumption and accelerate compute.

In [6]:
# 4. Set Up Training Arguments
import torch
from transformers import TrainingArguments

# Set Up Training Arguments for 20 Epochs
training_args = TrainingArguments(
    output_dir="./gpt2-tinystories112",
    eval_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,              # Set to 3 epochs
    logging_steps=5,                  # Log every 5 steps to keep track of loss progress
    save_strategy="epoch",            # Save a checkpoint at the end of every epoch
    save_total_limit=2,               # Keep only the last 2 checkpoints to avoid filling Colab disk
    fp16=torch.cuda.is_available(),   # Enables FP16 GPU acceleration if available
)

###6 Key Operations

* **Dataset Size Verification**: Prints total sample counts for `tokenized_train` and `tokenized_eval` to verify non-empty inputs before proceeding.
* **`Trainer(...)` Initialization**: Binds the loaded model, hyperparameters (`training_args`), tokenized datasets, and `data_collator` into a unified execution instance.
* **Batch Diagnostic Inspection**:
  * Extracts the first batch from `trainer.get_train_dataloader()`.
  * Logs the tensor shape of `input_ids`.
  * Checks `numel() == 0` to confirm that tensors contain active token data prior to execution.
* **`trainer.train()`**: Begins the fine-tuning loop across designated epochs if diagnostic checks pass.
* **Exception Handling**: Catches initialization or batch retrieval errors safely to avoid standard stack trace crashes.

In [7]:
# 5. Initialize Trainer & Train with Batch Diagnostics
print(f"Final Training samples: {len(tokenized_train)}")
print(f"Final Validation samples: {len(tokenized_eval)}")

if len(tokenized_train) > 0:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        data_collator=data_collator,
    )

    # Diagnostic: Check the first batch to ensure it's not empty
    try:
        train_dataloader = trainer.get_train_dataloader()
        it = iter(train_dataloader)
        first_batch = next(it)
        print(f"Diagnostic - First batch input_ids shape: {first_batch['input_ids'].shape}")

        if first_batch['input_ids'].numel() == 0:
            print("Error: Detected an empty batch! Tokenization or filtering is still failing.")
        else:
            print("Batch looks good. Starting training...")
            trainer.train()
    except Exception as e:
        print(f"Error during training initialization or first batch retrieval: {e}")
else:
    print("Error: The tokenized dataset is empty.")

Final Training samples: 429
Final Validation samples: 189
Diagnostic - First batch input_ids shape: torch.Size([8, 60])
Batch looks good. Starting training...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.555153,2.330446
2,2.171245,2.115411
3,2.197528,2.034749


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### 7 Key Operations

* **`checkpoint_path`**: Targets the exact epoch/step checkpoint directory generated during the `Trainer` execution run.
* **`from_pretrained(checkpoint_path)`**: Loads the saved model parameters, optimizer/scheduler metadata, and tokenizer config directly from local storage instead of downloading base models.
* **`model.to(device)`**: Moves model weights to CUDA (GPU) if available, falling back to CPU if no GPU hardware is detected.
* **`model.generate(...)` Configuration**:
  * **`max_new_tokens=100`**: Sets the maximum number of new tokens generated during autoregressive decoding.
  * **`do_sample=True`**: Enables probabilistic sampling for varied, non-deterministic text generation.
  * **`temperature=0.7`**: Adjusts token probability distribution sharpness to balance creativity and coherence.
  * **`pad_token_id=tokenizer.eos_token_id`**: Sets the padding token ID explicitly to the End-Of-Sequence (EOS) token ID to avoid warning logs during generation.
* **`tokenizer.decode(...)`**: Converts generated token IDs back into clean text format.

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Path directly to the checkpoint folder in your image
checkpoint_path = "/content/gpt2-tinystories112/checkpoint-162"

# Load the model and tokenizer directly from checkpoint-162
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForCausalLM.from_pretrained(checkpoint_path)

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Generate text
prompt = "Once upon a time, there was a little dog named"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Once upon a time, there was a little dog named Tim. Tim liked to play with his toys. One day, Tim decided to wear a dress to show off his new haircut. He had a big hat and a big collar. His mom took him to see his favorite toy, the bunny. Tim liked the hat because it was so small. He thought it looked cool and bright. He wanted to show off his new haircut! Tim wanted to show off his new hair and tie the tie. He was so happy and proud of himself! Tim was


In [13]:
# Generate text
prompt = "Once, there was a dragon named sheepstealer"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Once, there was a dragon named sheepstealer. He was very, very careful with his magic. One day, he saw something strange in the sky. He had a magic circle that could be used to pick the best spot to steal. The best spot to pick was the dark spot. He picked the spot with a big magic needle and then picked it with his magic wand. He picked the spot with his magic wand and never saw it again. The dragon was so happy! He had never seen anything like it before! He had never lost


In [14]:
# Generate text
prompt = "Once, there was a happy family of cats"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Once, there was a happy family of cats. The mother looked at the cat and said, "Hello, cat. Are you hungry?" The cat said, "No, I am hungry!" So, the mom said, "Let's go visit the shop and pick something up. We can buy some food." The cat liked the idea and decided to let the mom and dad go and clean the shop. The cat liked having a good time and wanted to eat more. So, they went to the shop and picked something to eat. They


### Download the Zip Fine-Tuned Model

In [16]:
import shutil

# Replace 'my_folder' with your folder path, and 'output_archive' with your desired zip name
folder_path = "/content/gpt2-tinystories112/checkpoint-162"
output_zip_name = "gpt2-tinystories-Model-Folder"  # Creates 'gpt2_checkpoint_backup.zip'

# Compress the directory
shutil.make_archive(output_zip_name, 'zip', folder_path)

print(f"Folder zipped successfully to {output_zip_name}.zip")

Folder zipped successfully to gpt2-tinystories-Model-Folder.zip
